# 01 · Ingestion pipeline
**What happens when a faculty member uploads a file:** parse → chunk → embed → store in Chroma.
Each step is shown separately, then run end-to-end exactly as the API does. Data written here is reused by notebooks 02–05.

In [1]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env  (set NB_OFFLINE=0 or edit below)
OFFLINE = os.getenv("NB_OFFLINE", "1") == "1"
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

mode            : OFFLINE (mock)
vector store    : local (/home/claude/proj/v1/rag_service/chroma_notebook)
embedding       : hash512
LLM providers   : ['gemini', 'groq', 'openai']


## 1. Create synthetic course files (PDF, PPTX, DOCX, XLSX, MD, CSV, HTML)

In [2]:
import synthetic
files = synthetic.make_all()
for subj, fs in files.items():
    print(subj, {n: f"{len(b)//1024 or 1} KB" for n, b in fs.items()})

BIO101 {'bio_notes.pdf': '2 KB', 'bio_slides.pptx': '27 KB'}
PHY101 {'physics.docx': '36 KB', 'physics_facts.xlsx': '4 KB'}
CS201 {'db_notes.md': '1 KB', 'db_facts.csv': '1 KB', 'db_page.html': '1 KB'}


## 2. Parse (page/slide/sheet aware)

In [3]:
from app.parsers import parse_bytes
secs = parse_bytes(files["BIO101"]["bio_notes.pdf"], "bio_notes.pdf")
for s in secs: print(f"page {s.page}:", s.text[:110].replace("\n"," "), "...")

page 1: Cell Biology Notes Photosynthesis occurs in the chloroplasts of plant cells. Chlorophyll absorbs light energy, ...
page 2: Genetics DNA replication is semi-conservative: each new double helix contains one original strand and one newl ...


## 3. Chunk (recursive splitter, provenance metadata, stable ids)

In [4]:
from app.chunker import chunk_sections
chunks = chunk_sections(secs, "demo-doc", {"subject_code":"BIO101","is_active":True,"file_name":"bio_notes.pdf"})
print(len(chunks), "chunks | sizes:", [len(c.page_content) for c in chunks])
print(json.dumps(chunks[0].metadata, indent=1))

2 chunks | sizes: [524, 369]
{
 "subject_code": "BIO101",
 "is_active": true,
 "file_name": "bio_notes.pdf",
 "document_id": "demo-doc",
 "chunk_index": 0,
 "page": 1,
 "heading": "",
 "chunk_id": "demo-doc:0"
}


## 4. Embed

In [5]:
from app.providers import get_embeddings
vecs = get_embeddings().embed_documents([c.page_content for c in chunks])
print("vectors:", len(vecs), "x", len(vecs[0]))

vectors: 2 x 512


## 5. Run the full pipeline for every synthetic file (this is what `POST /ingest` does)

In [6]:
from app.ingest import ingest_document, get_status
results = {}
for subj, fs in files.items():
    for name, data in fs.items():
        did = f"{subj}-{name}"
        results[did] = ingest_document(did, {"subject": subj, "is_active": True}, data=data, file_name=name)
import pandas as pd
pd.DataFrame(results).T[["chunkCount","durationMs","collection"]]

,chunkCount,durationMs,collection
BIO101-bio_notes.pdf,2,97,edu_chunks__hash512
BIO101-bio_slides.pptx,1,17,edu_chunks__hash512
PHY101-physics.docx,2,46,edu_chunks__hash512
PHY101-physics_facts.xlsx,1,14,edu_chunks__hash512
CS201-db_notes.md,1,10,edu_chunks__hash512
CS201-db_facts.csv,1,7,edu_chunks__hash512
CS201-db_page.html,1,41,edu_chunks__hash512


## 6. Verify what is really stored

In [7]:
from app.store import get_store
st = get_store()
print("total chunks in collection:", st.count())
print(get_status("BIO101-bio_notes.pdf"))
sample = st.chunks_of("PHY101-physics.docx")[0]
print(sample.metadata["subject_code"], "| page", sample.metadata["page"], "|", sample.page_content[:100])

total chunks in collection: 9
{'document_id': 'BIO101-bio_notes.pdf', 'metrics': {'characters_extracted': 893, 'sections_found': 2, 'chunks_created': 2, 'embeddings_created': 2, 'vectors_indexed': 2, 'embedding_model': 'hash512', 'duration_ms': 97, 'collection': 'edu_chunks__hash512'}, 'status': 'completed', 'step': 'done', 'error': None}
PHY101 | page 1 | Mechanics
Newton's second law states that the net force on an object equals its mass multiplied by i


## 7. Idempotency + failure handling

In [8]:
n0 = len(st.chunks_of("PHY101-physics.docx"))
ingest_document("PHY101-physics.docx", {"subject":"PHY101","is_active":True}, data=files["PHY101"]["physics.docx"], file_name="physics.docx")
assert len(st.chunks_of("PHY101-physics.docx")) == n0, "re-ingest must not duplicate"
from app.parsers import ParseError
try: ingest_document("bad", {"subject":"X"}, data=b"junk", file_name="bad.pdf")
except ParseError as e: print("clean failure ->", e, "| status:", get_status("bad")["status"])

invalid pdf header: b'junk'


EOF marker not found


EOF marker not found


clean failure -> Cannot open PDF: Stream has ended unexpectedly | status: failed
